# Dashboard pre analýzu Spread‑F javov 

Tento notebook slúži ako interaktívny nástroj na spracovanie a analýzu spektrogramov pomocou trénovaných segmentačných modelov.  
Po načítaní obrázka a modelu notebook:

- predikuje masku Spread‑F,
- extrahuje priemerné frekvenčné profily,
- deteguje jednotlivé Spread‑F javy,
- vykresľuje výsledky,
- generuje tabuľky a štatistiky,
- umožňuje spracovať jeden súbor alebo celý priečinok naraz.

---

##  Ako notebook používať

1. **Spusť všetky bunky notebooku**  
   - Je to nevyhnutné, aby sa načítali všetky funkcie, widgety a prepojili sa callbacky.

2. Po spustení sa zobrazí interaktívny dashboard:
   - výber priečinka s obrázkami,
   - výber modelu,
   - nastavenie prahovania a parametrov analýzy,
   - tlačidlá na spracovanie jedného obrázka alebo celého priečinka.

3. Výsledky sa zobrazia priamo pod dashboardom:
   - grafy,
   - overlay masky,
   - detegované eventy,
   - tabuľky s parametrami,
   - voliteľne export do CSV.



In [1]:
!pip install opencv-python

# Importy a základná konfigurácia dashboardu
V tejto bunke importujeme všetky potrebné knižnice a nastavíme základné globálne premenné:
- cesty k dátam (je nevyhnutne špecifikovať, správnu cestu k surovým dátam (.mat) a cestu k orezaným testovacím spektrogramom)
- cesty k modelom (je nevyhnutne špecifikovať, správnu cestu k modelom)
- veľkosť obrázkov
- zoznam modelových súborov


In [2]:
import numpy as np
import scipy.io as sio
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output
from skimage.measure import label, regionprops
import pandas as pd

#KONFIGURÁCIA
# It is important to enter the correct path to the .mat file (raw data)!!!
RAW_DATA_DIRS = [Path(f"data/RawData/zasielka{i}") for i in range(1, 14)]


# It is important to enter the correct path to the preprocessed data (cropped spectrograms), new data that the model did not see!!!
IMAGE_DIR = Path("DeepLabV3/test10DeepLabV3/images")
IMAGE_EXTS = (".png", ".jpg", ".jpeg")

# It is important to enter the correct path to the models!!!
MODEL_DIRS = [Path("selected_models_for_dashboard")]

IMG_SIZE = (384,384)

MODEL_FILES = []
for d in MODEL_DIRS:
    MODEL_FILES.extend(sorted(d.glob("*.keras")))

BATCH_RESULTS = []

2026-04-20 13:57:24.861716: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


# Funkcia na extrakciu priemerných profilov z masky

Táto funkcia `get_average_profiles`:
- nájde jednotlivé segmenty masky
- pre každý segment vypočíta priemernú frekvenciu v čase
- vráti zoznam profilov vhodných na ďalšiu analýzu


In [3]:
def get_average_profiles(mask, f_axis, t_axis):
    labeled = label(mask > 0)
    regions = sorted(regionprops(labeled), key=lambda r: r.centroid[0])
    
    profiles = []
    for region in regions:
        if region.area < 50:
            continue
        
        coords = region.coords
        t_min, t_max = coords[:, 1].min(), coords[:, 1].max()
        full_t_range = np.arange(t_min, t_max + 1)
        raw_f_indices = []
        
        for t_idx in full_t_range:
            f_at_t = coords[coords[:, 1] == t_idx][:, 0]
            if len(f_at_t) > 0:
                raw_f_indices.append(np.mean(f_at_t))
            else:
                raw_f_indices.append(np.nan)
        
        f_indices_arr = np.array(raw_f_indices)
        nans = np.isnan(f_indices_arr)
        if np.any(nans):
            x = lambda z: z.nonzero()[0]
            f_indices_arr[nans] = np.interp(x(nans), x(~nans), f_indices_arr[~nans])
        
        t_line = t_axis[full_t_range]
        safe_idx = np.clip(np.round(f_indices_arr).astype(int), 0, len(f_axis) - 1)
        f_line = f_axis[safe_idx]

        if len(t_line) > 40:
            profiles.append({"t_line": t_line, "f_line": f_line, "density": region.area})
       
    return profiles

# Analýza jednej priemernej krivky

Funkcia `analyze_average_curve`:
- vyhladzuje krivku
- počíta gradient
- hľadá tri bloky (nárast – pokles – nárast)
- extrahuje parametre Spread‑F javu


In [4]:
def analyze_average_curve(f_line, t_line, smooth_win=35, min_len=50, grad_pos_thresh=0.08, grad_neg_thresh=-0.06, min_block_len=10, min_amp=0.6, max_start_distance=40, max_end_distance=40):
    min_len = int(min_len)
    min_block_len = int(min_block_len)
    max_start_distance = int(max_start_distance)
    max_end_distance = int(max_end_distance)

    if len(f_line) < min_len:
        return []

    # Smoothing 
    kernel = np.ones(smooth_win) / smooth_win
    f_smooth = np.convolve(f_line, kernel, mode='same')
    grad = np.gradient(f_smooth, t_line)

    events = []
    i = 0
    n = len(grad)
    last_end_idx = -1

    while i < n:

        # BLOCK A — FIRST INCREASE
        while i < n and grad[i] <= grad_pos_thresh:
            i += 1
        if i >= n:
            break
        rise1_start = i

        limit = int(min(i + max_end_distance, n))
        while i < limit and grad[i] > grad_pos_thresh:
            i += 1
        rise1_end = i - 1

        if rise1_end - rise1_start + 1 < min_block_len:
            continue
        if f_smooth[rise1_end] - f_smooth[rise1_start] < min_amp:
            continue

        # BLOCK B — DROP
        while i < n and grad[i] >= grad_neg_thresh:
            i += 1
        if i >= n:
            break
        fall_start = i

        limit = int(min(i + max_end_distance, n))
        while i < limit and grad[i] < grad_neg_thresh:
            i += 1
        fall_end = i - 1

        if fall_end - fall_start + 1 < min_block_len:
            continue
        if f_smooth[fall_start] - f_smooth[fall_end] < min_amp:
            continue

        # BLOCK C — SECOND INCREASE
        while i < n and grad[i] <= grad_pos_thresh:
            i += 1
        if i >= n:
            break
        rise2_start = i

        limit = int(min(i + max_end_distance, n))
        while i < limit and grad[i] > grad_pos_thresh:
            i += 1
        rise2_end = i - 1

        if rise2_end - rise2_start + 1 < min_block_len:
            continue
        if f_smooth[rise2_end] - f_smooth[rise2_start] < min_amp:
            continue

        # PEAK & TROUGH
        local_peak_idx = rise1_start + np.argmax(f_smooth[rise1_start:fall_start+1])
        local_trough_idx = fall_start + np.argmin(f_smooth[fall_start:rise2_start+1])

        # START INDEX
        idx_start = local_peak_idx
        limit = int(max(local_peak_idx - max_start_distance, 0))
        if last_end_idx >= 0:
            limit = max(limit, last_end_idx + 1)

        for k in range(limit, local_peak_idx):
            if grad[k] > grad_pos_thresh:
                idx_start = k
                break

        # END INDEX
        idx_end = rise2_end
        limit = int(min(rise2_end + max_end_distance, n))

        for k in range(rise2_end + 1, limit):
            if grad[k] < grad_pos_thresh:
                idx_end = k
                break

        # Final event parameters
        slope_val = (f_smooth[local_trough_idx] - f_smooth[local_peak_idx]) / (
            t_line[local_trough_idx] - t_line[local_peak_idx] + 1e-6
        )
        drop_val = f_smooth[local_peak_idx] - f_smooth[local_trough_idx]

        events.append({
            "start_t": t_line[idx_start],
            "start_f": f_line[idx_start],
            "peak_t": t_line[local_peak_idx],
            "peak_f": f_line[local_peak_idx],
            "trough_t": t_line[local_trough_idx],
            "trough_f": f_line[local_trough_idx],
            "end_t": t_line[idx_end],
            "end_f": f_line[idx_end],
            "slope": slope_val,
            "drop": drop_val
        })

        last_end_idx = idx_end
        i = idx_end + 1

    return events

# Hlavná funkcia na spracovanie jedného obrázka

Funkcia `run_analysis`:
- načíta obrázok
- načíta model a predikuje masku
- aplikuje prahovanie (alebo Otsu)
- vykoná morfologické operácie
- nájde zodpovedajúci .mat súbor (ak existuje)
- extrahuje profily
- vykoná analýzu Spread‑F
- vykreslí výsledky
- pripraví tabuľky a štatistiky


In [5]:
def run_analysis(model_path, img_path, thr, morph, min_amp, max_start_distance, max_end_distance, min_len, min_block_len, f_min, f_max, save_csv=False, use_full_frange=False, use_otsu=False):
    # Load image
    img_raw = cv2.imread(str(img_path))
    if img_raw is None:
        print(f"PNG failed to load: {img_path}")
        return

    img_rgb = cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB)

    # Load model and predict mask
    model = tf.keras.models.load_model(model_path, compile=False)
    img_input = cv2.resize(img_rgb, IMG_SIZE) / 255.0
    pred = model.predict(img_input[None, ...], verbose=0)[0, :, :, 0]

    # Confidence metrics
    mean_pred = float(pred.mean())
    
    mask_pixels = pred[pred > 0.1]
    mask_conf = float(mask_pixels.mean()) if len(mask_pixels) > 0 else 0.0
    
    eps = 1e-6
    entropy = float(-np.mean(pred * np.log(pred + eps) + (1 - pred) * np.log(1 - pred + eps)))


    if use_otsu:
        pred_uint8 = (pred * 255).astype(np.uint8)
        otsu_thr, otsu_mask = cv2.threshold(pred_uint8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        mask_full = cv2.resize((otsu_mask > 0).astype(np.uint8),
                               (img_rgb.shape[1], img_rgb.shape[0]),
                               interpolation=cv2.INTER_NEAREST)
        final_threshold = otsu_thr / 255.0  
    else:
        mask_full = cv2.resize((pred > thr).astype(np.uint8),
                               (img_rgb.shape[1], img_rgb.shape[0]),
                               interpolation=cv2.INTER_NEAREST)
        final_threshold = thr

    # Save binary mask if checkbox is active 
    if save_mask_checkbox.value:
        mask_name = Path(img_path).stem + "_predicted_mask.npy"
        np.save(mask_name, mask_full.astype(np.uint8))
        print(f"Mask saved: {mask_name}")

    
    if morph != 0:
        kernel = np.ones((3, 3), np.uint8)
        if morph > 0:
            for _ in range(morph):
                mask_full = cv2.dilate(mask_full, kernel, iterations=1)
        else:
            for _ in range(-morph):
                mask_full = cv2.erode(mask_full, kernel, iterations=1)

    # Try to find matching .mat file
    base = img_path.stem.split("_crop")[0]
    if base.startswith("s") and not base[1:].isdigit():
        base = base[1:]

    mat_p = None
    for d in RAW_DATA_DIRS:
        candidates = list(d.glob(f"*{base}*.mat"))
        if candidates:
            mat_p = candidates[0]
            break

    # If .mat DOES NOT exist → pixel mode with normalization
    if mat_p is None:
        print(f"I didn't find any .mat for the prefix '{base}'.")
        print("The analysis will be performed in pixel coordinates!")

        img_h, img_w = img_rgb.shape[:2]
        
        f_axis_phys = np.linspace(img_h, 0, img_h)  
        t_axis_phys = np.linspace(0, img_w, img_w)

        # Normalization of parameters
        orig_df = 20.0
        orig_dt = 180.0
        
        scale_f = img_h / orig_df
        scale_t = img_w / orig_dt
        
        grad_pos = grad_pos_slider.value * (scale_f / scale_t)
        grad_neg = grad_neg_slider.value * (scale_f / scale_t)
        min_amp_px = min_amp_slider.value * scale_f
        max_start_px = max_start_slider.value 
        max_end_px = max_end_slider.value 
        
        # Profile extraction
        profiles = get_average_profiles(mask_full, f_axis_phys, t_axis_phys)

        # Visualization
        fig, axs = plt.subplots(1, 2, figsize=(14, 6))
    
        axs[0].imshow(img_rgb, extent=[0, img_w, 0, img_h], aspect="auto")
        axs[0].set_title("Spectrogram (pixels)", fontsize=20, fontweight='bold')
        axs[0].tick_params(axis='both', labelsize=14)
        axs[0].set_xlabel("time (px)", fontsize=16)
        axs[0].set_ylabel("Δf (px)", fontsize=16)
                
        
        axs[1].imshow(mask_full, extent=[0, img_w, 0, img_h], aspect="auto", cmap="gray")
        if morph != 0:
            axs[1].set_title(f"Mask (threshold: {round(final_threshold,2)}, Mask thickness={morph})", fontsize=20, fontweight='bold')
        else:
            axs[1].set_title(f"Mask (Threshold={round(final_threshold,2)})", fontsize=20, fontweight='bold')
        axs[1].tick_params(axis='both', labelsize=14)
        axs[1].set_xlabel("time (px)", fontsize=16)
        axs[1].set_ylabel("Δf (px)", fontsize=16)
    
    
        plt.tight_layout()
        plt.show()
    
        # Overlay + analysis
        fig, ax = plt.subplots(figsize=(14, 8))
        ax.imshow(img_rgb, extent=[0, img_w, 0, img_h], aspect="auto")
    
    
        overlay = np.dstack([
            mask_full * 255,
            np.zeros_like(mask_full),
            np.zeros_like(mask_full),
            mask_full * 80
        ])
        ax.imshow(overlay, extent=[0, img_w, 0, img_h], aspect="auto", interpolation="nearest")
        ax.set_yticks(np.arange(0, img_h+1, 500))


        results_list = []

        html_table = f"<h3>{Path(img_path).stem}</h3><table border='1' style='width:100%; text-align:center; border-collapse:collapse;'>"
        html_table += "<tr style='background-color:#2C3E50; color:white;'><th>Frequence</th><th>Spread-F</th><th>Start [px]</th><th>Peak [px]</th><th>Trough [px]</th><th>End [px]</th><th>Slope</th><th>Drop</th></tr>"

        for i, prof in enumerate(profiles):
            ax.plot(prof["t_line"], prof["f_line"], color='cyan', lw=1.5, alpha=0.9)

            events = analyze_average_curve(
                prof["f_line"], prof["t_line"],
                min_amp=min_amp_px,
                max_start_distance=max_start_px,
                max_end_distance=max_end_px,
                min_len=min_len_slider.value,
                min_block_len=min_block_slider.value,
                grad_pos_thresh=grad_pos,
                grad_neg_thresh=grad_neg
            )

            for j, res in enumerate(events, start=1):
                ax.scatter(res["start_t"], res["start_f"], color='lime', s=100, edgecolors='black')
                ax.scatter(res["peak_t"], res["peak_f"], color='yellow', s=120, edgecolors='black')
                ax.scatter(res["trough_t"], res["trough_f"], color='saddlebrown', s=120, edgecolors='black')
                ax.scatter(res["end_t"], res["end_f"], color='red', s=100, edgecolors='black')

                ax.annotate('', xy=(res["trough_t"], res["trough_f"]),
                            xytext=(res["peak_t"], res["peak_f"]),
                            arrowprops=dict(arrowstyle='->', color='yellow', lw=2.5))

                ax.text(res["start_t"], res["start_f"] + 5,
                        f"F{i+1}.{j}", color='white', weight='bold', fontsize=12)

                html_table += (
                    f"<tr><td>F{i+1}</td>"
                    f"<td>{j}</td>"
                    f"<td>{res['start_t']:.1f}</td>"
                    f"<td>{res['peak_t']:.1f}</td>"
                    f"<td>{res['trough_t']:.1f}</td>"
                    f"<td>{res['end_t']:.1f}</td>"
                    f"<td>{res['slope']:.4f}</td>"
                    f"<td>{res['drop']:.3f}</td></tr>"
                )

                results_list.append({
                    "Frequence": f"F{i+1}",
                    "Event": j,
                    "Start_px": round(res["start_t"], 1),
                    "Peak_px": round(res["peak_t"], 1),
                    "Trough_px": round(res["trough_t"], 1),
                    "End_px": round(res["end_t"], 1),
                    "Slope": round(res["slope"], 4),
                    "Drop": round(res["drop"], 3)
                })

        ax.set_title("Analysis of spread-F (pixel axes)",  fontsize=20, fontweight='bold')
        ax.set_xlabel("time (px)")
        ax.set_ylabel("Δf (px)")
        ax.tick_params(axis='both', labelsize=14)   
        ax.set_xlabel(ax.get_xlabel(), fontsize=16) 
        ax.set_ylabel(ax.get_ylabel(), fontsize=16)

        plt.tight_layout()
        plt.show()

    
    elif mat_p:
        # If .mat EXISTS → physical mode (original)
        data_mat = sio.loadmat(mat_p)
        t_orig = data_mat["timeaxis"].squeeze()
        f_orig = data_mat["faxis"].squeeze()
    
        if use_full_frange:
            f_crop = f_orig
        else:
            f_idx = np.where((f_orig >= f_min) & (f_orig <= f_max))[0]
            f_crop = f_orig[f_idx]
    
        f_axis_phys = np.linspace(f_crop.max(), f_crop.min(), img_rgb.shape[0])
        t_axis_phys = np.linspace(t_orig.min(), t_orig.max(), img_rgb.shape[1])
    
        profiles = get_average_profiles(mask_full, f_axis_phys, t_axis_phys)

        # Spectrogram and mask (side by side)
        fig, axs = plt.subplots(1, 2, figsize=(14, 6))
        extent = [t_orig.min(), t_orig.max(), f_axis_phys[-1], f_axis_phys[0]]
        
        axs[0].imshow(img_rgb, extent=extent, aspect="auto")
        axs[0].set_title("Spectrogram", fontsize=20, fontweight='bold')
        axs[0].tick_params(axis='both', labelsize=14)
        axs[0].set_xlabel("time (min)", fontsize=16)
        axs[0].set_ylabel("Δf (Hz)", fontsize=16)
        
        axs[1].imshow(mask_full, extent=extent, aspect="auto", cmap="gray")
        if morph != 0:
            axs[1].set_title(f"Mask (threshold: {round(final_threshold,2)}, Mask thickness={morph})", fontsize=20, fontweight='bold')
        else:
            axs[1].set_title(f"Mask (threshold: {round(final_threshold,2)})", fontsize=20, fontweight='bold')
        axs[1].tick_params(axis='both', labelsize=14)
        axs[1].set_xlabel("time (min)", fontsize=16)
        axs[1].set_ylabel("Δf (Hz)", fontsize=16)

        
        plt.tight_layout()
        plt.show()
    
        # Main analysis (overlay + curves) 
        fig, ax = plt.subplots(figsize=(14, 8))
        
        ax.imshow(img_rgb, extent=extent, aspect="auto")
        
        overlay = np.dstack([
            mask_full * 255,
            np.zeros_like(mask_full),
            np.zeros_like(mask_full),
            mask_full * 80
        ])
        ax.imshow(overlay, extent=extent, aspect="auto", interpolation="nearest")
      
        results_list = []
    
    
        html_table = f"<h3>{Path(mat_p).stem}</h3><table border='1' style='width:100%; text-align:center; border-collapse:collapse;'>"
        html_table += "<tr style='background-color:#2C3E50; color:white;'><th>Frequence</th><th>Spread-F</th><th>Start [min]</th><th>Peak [min]</th><th>Trough [min]</th><th>End [min]</th><th>Slope [Hz/min]</th><th>Drop [Hz]</th></tr>"
    
        for i, prof in enumerate(profiles):
            ax.plot(prof["t_line"], prof["f_line"], color='cyan', lw=1.5, alpha=0.9)
    
            events = analyze_average_curve(
                prof["f_line"], prof["t_line"],
                min_amp=min_amp,
                max_start_distance=max_start_distance,
                max_end_distance=max_end_distance,
                min_len=min_len,
                min_block_len=min_block_len,
                grad_pos_thresh=grad_pos_slider.value,
                grad_neg_thresh=grad_neg_slider.value
            )
    
            for j, res in enumerate(events, start=1):
                ax.scatter(res["start_t"], res["start_f"], color='lime', s=100,
                           edgecolors='black', zorder=5, label='Start')
                ax.scatter(res["peak_t"], res["peak_f"], color='yellow', s=120,
                           edgecolors='black', zorder=6, label='Peak')
                ax.scatter(res["trough_t"], res["trough_f"], color='saddlebrown', s=120,
                           edgecolors='black', zorder=6, label='Trough')
                ax.scatter(res["end_t"], res["end_f"], color='red', s=100,
                           edgecolors='black', zorder=5, label='End')
    
                ax.annotate(
                    '',
                    xy=(res["trough_t"], res["trough_f"]),
                    xytext=(res["peak_t"], res["peak_f"]),
                    arrowprops=dict(arrowstyle='->', color='yellow', lw=2.5)
                )
                ax.text(
                    res["start_t"],
                    res["start_f"] + 0.5,
                    f"F{i+1}.{j}",
                    color='white',
                    weight='bold',
                    fontsize=12
                )

    
                html_table += (
                    f"<tr><td>F{i+1}</td>"
                    f"<td>{j}</td>"
                    f"<td>{res['start_t']:.2f}</td>"
                    f"<td>{res['peak_t']:.2f}</td>"
                    f"<td>{res['trough_t']:.2f}</td>"
                    f"<td>{res['end_t']:.2f}</td>"
                    f"<td>{res['slope']:.4f}</td>"
                    f"<td>{res['drop']:.3f}</td></tr>"
                )
    
                results_list.append({
                    "Frequence": f"F{i+1}",
                    "Event": j,
                    "Start_min": round(res["start_t"], 2),
                    "Peak_min": round(res["peak_t"], 2),
                    "Trough_min": round(res["trough_t"], 2),
                    "End_min": round(res["end_t"], 2),
                    "Slope_Hz_per_min": round(res["slope"], 4),
                    "Drop_Hz": round(res["drop"], 3)
                })
        
        handles, labels = ax.get_legend_handles_labels()
        unique = dict(zip(labels, handles))
        ax.legend(unique.values(), unique.keys(), fontsize=12, loc='upper right')
        
        ax.set_title("Analysis of spread-F", fontsize=20, fontweight='bold')
        ax.set_xlabel("time (min)")
        ax.set_ylabel("Δf (Hz)")
        ax.tick_params(axis='both', labelsize=14)   
        ax.set_xlabel(ax.get_xlabel(), fontsize=16) 
        ax.set_ylabel(ax.get_ylabel(), fontsize=16)

    
        plt.tight_layout()
        plt.show()

    
    html_table += "</table>"

    
    num_profiles = len(profiles)
    
    DENSITY_THRESHOLD = 60000
    
   # Finding the number of leads at the beginning of the mask 
    if mat_p is None:
        start_slice = mask_full[:, :5]
    else:
        t_limit = 0.2
        px_limit = np.searchsorted(t_axis_phys, t_axis_phys[0] + t_limit)
        start_slice = mask_full[:, :px_limit]
    
    labeled_start = label(start_slice > 0)
    num_start_leads = len(regionprops(labeled_start))
    
    # Finding the number of profiles that start in the first 5 min/px
    num_profiles_start = 0
    for prof in profiles:
        if prof["t_line"][0] <= 0.2:
            num_profiles_start += 1

    # Missing spread-F
    missing_spread = "False"
    # Missing lead
    missing_lead = num_profiles_start < num_start_leads
    
    
    # Detecting dense profiles 
    dense_profiles = [p for p in profiles if p.get("density", 0) > DENSITY_THRESHOLD]

    num_dense = len(dense_profiles)
    
    # Difficulty calculation
    if num_profiles == 1:
        difficulty = 3 if num_dense == 1 else 1
    
    elif num_profiles == 2:
        difficulty = 2 if num_dense >= 1 else 1
    
    elif num_profiles <= 3:
        difficulty = 1
    
    elif num_profiles <= 6:
        difficulty = 2
    
    else:
        difficulty = 3

    
    if difficulty == 1 and len(results_list) > 0:
        # Number of lines
        num_profiles = len(profiles)
    
        # Prepare an event counter for each line
        line_event_counts = {f"F{i+1}": 0 for i in range(num_profiles)}
    
        # We go through all detected events and assign them to lines
        for r in results_list:
            freq = r["Frequence"]   # napr. "F1"
            line_event_counts[freq] += 1
    
        # We get the number of events in each line
        values = list(line_event_counts.values())
    
        # If they are not all the same → there is a line with a different number of Spread-F events
        if len(set(values)) > 1:
            missing_spread = "True"
    
    # If lead is missing, Spread-F cannot be missing   
    if missing_lead:
        missing_spread = "False"


    html_table_profiles = (
        "<h3>Additional informations</h3>"
        "<table border='1' style='width:40%; text-align:center; border-collapse:collapse;'>"
        "<tr style='background-color:#2C3E50; color:white;'><th>Parameter</th><th>Value</th></tr>"
        f"<tr><td>Number of profiles</td><td>{num_profiles}</td></tr>"
        f"<tr><td>Difficulty level</td><td>{difficulty}</td></tr>"
        f"<tr><td>Missing lead</td><td>{missing_lead}</td></tr>"
        f"<tr><td>Missing Spread-F</td><td>{missing_spread}</td></tr>"
        f"<tr><td>Mask confidence</td><td>{mask_conf:.4f}</td></tr>"
        "</table>"
    )


    display(
        widgets.HTML(
            f"""
            <div style='display:flex; gap:40px;'>
                <div style='flex:3; border:1px solid #ccc; padding:10px; border-radius:8px;'>
                    {html_table}
                </div>
                <div style='flex:1; border:1px solid #ccc; padding:10px; border-radius:8px; background-color:#f9f9f9;'>
                    {html_table_profiles}
                </div>
            </div>
            """
        )
    )

    if save_csv and len(results_list) > 0:
        for r in results_list:
            r.update({
                "Number_of_profiles": num_profiles,
                "Difficulty_level": difficulty,
                "Missing lead": missing_lead,
                "Missing Spread-F": missing_spread,
                "Mask_confidence": round(mask_conf, 4)
            })
    
        df = pd.DataFrame(results_list)
        csv_name = Path(mat_p).stem + ".csv"
        df.to_csv(csv_name, index=False, encoding="utf-8")
        print(f"Results saved to CSV: {csv_name}")
    elif save_csv and len(results_list) == 0 and mat_p is None:
        results_list.append({
            "Number_of_profiles": num_profiles,
            "Difficulty_level": difficulty,
            "Missing lead": missing_lead,
            "Missing Spread-F": missing_spread,
            "Mask_confidence": round(mask_conf, 4)
        })
        df = pd.DataFrame(results_list)
        if mat_p:
            csv_name = Path(mat_p).stem + ".csv"
        else:
            csv_name = Path(img_path).stem + ".csv" 
        df.to_csv(csv_name, index=False, encoding="utf-8")
        print(f"Results saved to CSV: {csv_name}")
        
        
    # Saving results for batch export
    record_base = {
        "Name": Path(mat_p).stem if mat_p else Path(img_path).stem,
        "Missing lead": missing_lead,
        "Missing Spread-F": missing_spread,
        "Mask confidence": round(mask_conf, 4)
    }
    if len(results_list) == 0:
        BATCH_RESULTS.append(record_base)
        
    if mat_p is None:
        # each event as a separate line for mat_p is None
        for r in results_list:
            rec = record_base.copy()
            rec.update({
                "Frequence": r.get("Frequence", ""),
                "Spread-F": r.get("Event", ""),
                "Start": r.get("Start_px", ""),
                "Peak": r.get("Peak_px", ""),
                "Trough": r.get("Trough_px", ""),
                "End": r.get("End_px", ""),
                "Slope": r.get("Slope", ""),
                "Drop": r.get("Drop", "")
            })
            BATCH_RESULTS.append(rec)
    else:
        # each event as a separate line for mat_p 
        for r in results_list:
            rec = record_base.copy()
            rec.update({
                "Frequence": r.get("Frequence", ""),
                "Spread-F": r.get("Event", ""),
                "Start": r.get("Start_min", ""),
                "Peak": r.get("Peak_min", ""),
                "Trough": r.get("Trough_min", ""),
                "End": r.get("End_min", ""),
                "Slope": r.get("Slope_Hz_per_min", ""),
                "Drop": r.get("Drop_Hz", "")
            })
            BATCH_RESULTS.append(rec)

# Batch spracovanie všetkých obrázkov v priečinku

Funkcia `run_batch_analysis`:
- prejde všetky obrázky v zvolenom priečinku
- pre každý zavolá `run_analysis`
- výsledky ukladá do globálneho zoznamu BATCH_RESULTS definovaného na začiatku


In [6]:
def run_batch_analysis(model_path, thr, morph, min_amp, max_start_distance, max_end_distance, min_len, min_block_len, f_min, f_max, save_csv=False, use_full_frange=False, use_otsu=False):

    image_files = list_images()

    if not image_files:
        print("No images found in the folder!")
        return

    for img_path in image_files:
        out = widgets.Output()  
        display(out)

        with out:
            print(f"I process: {img_path.name}")

            run_analysis(Path(model_path), Path(img_path), thr, morph, min_amp, max_start_distance, max_end_distance, min_len, min_block_len, f_min, f_max, save_csv, use_full_frange, use_otsu)

# Pomocné funkcie na listovanie obrázkov

Funkcie `list_images` a `update_image_dropdown`:
- načítajú obrázky z priečinka
- aktualizujú dropdown widget pri zmene priečinka


In [7]:
image_dir_text = widgets.Text(value=str(IMAGE_DIR), layout={'width': '80%'})

def list_images():
    img_dir = Path(image_dir_text.value)
    if not img_dir.exists():
        return []
    return sorted([p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS])

def update_image_dropdown(change):
    files = list_images()
    image_select.options = [(p.name, str(p)) for p in files]

# Nastavenie widgetov pre interaktívne ovládanie

V tejto časti sa vytvára:
- výber modelu
- výber obrázka
- slidery pre parametre analýzy
- tlačidlá na spustenie


In [8]:
IMAGE_FILES = list_images()
image_select = widgets.Dropdown(options=[(p.name, str(p)) for p in IMAGE_FILES], layout={'width': '80%'})
image_dir_text.observe(update_image_dropdown, names='value')
model_select = widgets.Dropdown(options=[(m.name, str(m)) for m in MODEL_FILES])
thr_slider = widgets.FloatSlider(value=0.45, min=0.1, max=0.9, step=0.05)
use_otsu_checkbox = widgets.Checkbox(value=False, indent=False)
mask_morph_slider = widgets.IntSlider(value=0, min=-50, max=10, step=1)

use_full_frange_checkbox = widgets.Checkbox(value=False, indent=False)
fmin_slider = widgets.FloatSlider(value=-10, min=-50, max=0, step=0.5)
fmax_slider = widgets.FloatSlider(value=10, min=0, max=50, step=0.5)

grad_pos_slider = widgets.FloatSlider(value=0.04, min=0.01, max=0.2, step=0.005)
grad_neg_slider = widgets.FloatSlider(value=-0.04, min=-0.2, max=-0.01, step=0.005)
min_amp_slider = widgets.FloatSlider(value=0.5, min=0.0, max=2.0, step=0.05)
min_len_slider = widgets.IntSlider(value=50, min=10, max=200, step=5)
min_block_slider = widgets.IntSlider(value=10, min=5, max=100, step=1)
max_start_slider = widgets.IntSlider(value=70, min=5, max=200, step=5)
max_end_slider = widgets.IntSlider(value=40, min=5, max=200, step=5)
save_csv_checkbox = widgets.Checkbox(value=False, indent=False)
save_mask_checkbox = widgets.Checkbox(value=False, indent=False)


batch_btn = widgets.Button(description="Process entire folder", button_style="info")
analyze_btn = widgets.Button(description="Process one file", button_style="success")
export_btn = widgets.Button(description="Export all results", button_style="warning")
ui_output = widgets.Output()

# Prepojenie widgetov s funkciami

Definicia callbackov pre tlačidlá:
- Process one file
- Process entire folder
- Export results


In [9]:
def on_batch_click(_):
    BATCH_RESULTS.clear()  
    with ui_output:
        run_batch_analysis(model_select.value, thr_slider.value, mask_morph_slider.value, min_amp_slider.value, max_start_slider.value, max_end_slider.value, min_len_slider.value,
            min_block_slider.value, fmin_slider.value, fmax_slider.value, save_csv_checkbox.value, use_full_frange_checkbox.value, use_otsu_checkbox.value)

batch_btn.on_click(on_batch_click)

def on_click(_):
    BATCH_RESULTS.clear()  
    with ui_output:
        clear_output(wait=True)
        run_analysis(Path(model_select.value), Path(image_select.value), thr_slider.value, mask_morph_slider.value, min_amp_slider.value, max_start_slider.value, max_end_slider.value,
            min_len_slider.value, min_block_slider.value, fmin_slider.value, fmax_slider.value, save_csv_checkbox.value, use_full_frange_checkbox.value, use_otsu_checkbox.value)


analyze_btn.on_click(on_click)

def on_export_click(_):
    if not BATCH_RESULTS:
        print("First, start batch processing!")
        return

    df = pd.DataFrame(BATCH_RESULTS)

    # column sorting
    cols = ["Name", "Frequence", "Spread-F", "Start", "Peak", "Trough", "End", "Slope", "Drop", "Missing lead", "Missing Spread-F", "Mask confidence"]
    for c in cols:
        if c not in df.columns:
            df[c] = ""

    df = df[cols]
    
    df.to_csv("batch_results.csv", index=False, encoding="utf-8")
    print("Results saved to batch_results.csv")
    
export_btn.on_click(on_export_click)

# Zobrazenie používateľského rozhrania

Zobrazenie všetkých widgetov a príprava výstupného okna.


In [10]:
block_input = widgets.VBox(
    [
        widgets.HTML("<h2 style='color:#1f77b4'>Input and model prediction</h2>"),

        widgets.HBox([widgets.HTML("<b>Image folder</b>"), image_dir_text]),
        widgets.HBox([widgets.HTML("<b>Image</b>"), image_select]),
        widgets.HBox([widgets.HTML("<b>Model</b>"), model_select]),

        widgets.HBox([
            widgets.VBox([widgets.HTML("<b>Threshold</b>"), thr_slider]),
            widgets.VBox([widgets.HTML("<b>Mask thickness</b>"), mask_morph_slider]),
            widgets.VBox([widgets.HTML("<b>Otsu threshold</b>"), use_otsu_checkbox]),
        ]),

        widgets.HBox([
            widgets.VBox([widgets.HTML("<b>f_min:</b>"), fmin_slider]),
            widgets.VBox([widgets.HTML("<b>f_max:</b>"), fmax_slider]),
            widgets.VBox([widgets.HTML("<b>Use full range:</b>"), use_full_frange_checkbox]),
        ]),
    ],
    layout=widgets.Layout(border="2px solid #1f77b4", padding="10px", margin="10px 0")
)



block_analysis = widgets.VBox(
    [
        widgets.HTML("<h2 style='color:#1f77b4'>Analysis parameters</h2>"),

        widgets.HBox([
            widgets.VBox([widgets.HTML("<b>grad_positive</b>"), grad_pos_slider]),
            widgets.VBox([widgets.HTML("<b>grad_negative</b>"), grad_neg_slider]),
        ]),

        widgets.HBox([
            widgets.VBox([widgets.HTML("<b>min_len</b>"), min_len_slider]),
            widgets.VBox([widgets.HTML("<b>min_block</b>"), min_block_slider]),
            widgets.VBox([widgets.HTML("<b>min_amplitude</b>"), min_amp_slider]),
        ]),

        widgets.HBox([
            widgets.VBox([widgets.HTML("<b>max_start</b>"), max_start_slider]),
            widgets.VBox([widgets.HTML("<b>max_end</b>"), max_end_slider]),
            widgets.VBox([widgets.HTML("<b>Save results to CSV</b>"), save_csv_checkbox]),
            widgets.VBox([widgets.HTML("<b>Save binary mask</b>"), save_mask_checkbox]),
        ]),
    ],
    layout=widgets.Layout(border="2px solid #1f77b4", padding="10px", margin="10px 0")
)

display(widgets.VBox([block_input, block_analysis, widgets.HBox([analyze_btn, batch_btn, export_btn]), ui_output]))